# 基礎編14
このノートブックでは、delegationによるサブコントラクト呼び出しとDGALによる制限の例
を示します。

## 概要

スマートコントラクトの中から、別のスマートコントラクトをdelegation(代理)で呼び出すことができます。  
delegationの有無によって、呼び出されたサブコントラクトのcallerが変化します。  
delegationの場合、サブコントラクトのcallerは、呼び出し元のコントラクトのcallerになります。  
delegationではない場合、サブコントラクトのcallerは、呼び出し元のコントラクトになります。  
デフォルトでは、delegationが無制限に許可されていますが、DGALを指定することで、これを制限することができます。  
DGALは、delegationで呼び出すことができるサブコントラクトのホワイトリストです。


## 準備

In [1]:
var { adminWallet, rpc, deploySmartContract } = await import('../lib/notebook-util.mjs');

## スマートコントラクトのデプロイ

callerIdを返す単純なコントラクトをデプロイします。

In [2]:
var subcid = await deploySmartContract(function basic14sub(a){
    return getCallerId();
});
console.log(subcid);

c075525704


サブコントラクトをdelegationで呼び出すコントラクトをデプロイします。  
(__subcid__はサブコントラクトのIDに置換されます)

In [3]:
var cid_dlg = await deploySmartContract(function basic14(){
    var cnt = openContract('__subcid__', { delegation: true }); // サブコントラクトをdelegationで開きます
    var res = cnt.call(); // サブコントラクトを呼び出します
    return [getCallerId(), res]; // 元のコントラクトのcallerと、サブコントラクトのcallerを返します
}, {
    replacers: [['__subcid__', subcid ]]
});
console.log(cid_dlg);

c088552559


比較のため、サブコントラクトを普通に呼び出すコントラクトをデプロイします。  

In [4]:
var cid_nondlg = await deploySmartContract(function basic14_nondlg(){
    var cnt = openContract('__subcid__'); // サブコントラクトを開きます
    var res = cnt.call(); // サブコントラクトを呼び出します
    return [getCallerId(), res]; // 元のコントラクトのcallerと、サブコントラクトのcallerを返します
}, {
    replacers: [['__subcid__', subcid ]]
});
console.log(cid_nondlg);

c026290095


サブコントラクトを実行できるように権限の設定をします。

In [5]:
await rpc.call(adminWallet, 'c1update', { id: subcid, prop: 'add accessible_to', value: [ cid_nondlg ] });

{
  txno: 701926,
  txid: 'x62wi9nC2z6G5pVJbVm59rYFsjK84kmAZTK2oR4HfJHHQB',
  status: 'ok',
  value: null
}

## サブコントラクト実行時のcallerの違い

delegationの場合、サブコントラクトのcallerは、呼び出し元のコントラクトのcallerになります。

In [6]:
var resp = await rpc.call(adminWallet, cid_dlg);
console.log(resp);

{
  txno: 701927,
  txid: 'xoYWPn7whZCSWnWzEgSwWYcwF2Qr69WTDmyUPWPsNcY66',
  status: 'ok',
  value: [ 'u58281888', 'u58281888' ]
}


delegationではない場合、サブコントラクトのcallerは、呼び出し元のコントラクトになります。

In [7]:
var resp = await rpc.call(adminWallet, cid_nondlg);
console.log(resp);

{
  txno: 701929,
  txid: 'xiyLq6Rcedqd9T5A66i7x3SDB8e9xadoXoo7bNmZJiUZn',
  status: 'ok',
  value: [ 'u58281888', 'c026290095' ]
}


## delegationの禁止

デフォルトでは、delegationが無制限に許可されています。  
これを禁止するには、DGALに空の配列を指定してから実行します。  
delegationが禁止されているため、delegationでのサブコントラクト呼び出しの際にエラーとなります。

In [8]:
var resp = await rpc.call(adminWallet, cid_dlg, {}, { DGAL: [] });
console.log(resp);

{
  txno: 701931,
  txid: 'x88XmfW9PENxenwLeM6Uoc2eHt4ne2tt8LuGJxgkZLQcw',
  status: 'aborted',
  value: 'TypeError: delegation not granted\n    at c088552559:1:13'
}


DGALの指定は、delegationではないサブコントラクト呼び出しには影響しません。

In [9]:
var resp = await rpc.call(adminWallet, cid_nondlg, {}, { DGAL: [] });
console.log(resp);

{
  txno: 701932,
  txid: 'xvrJvVH5gBLsZXjep84Dn5znYqAD663FEzt32uuDTm5bn',
  status: 'ok',
  value: [ 'u58281888', 'c026290095' ]
}


## delegationの許可

DGALにリストすることにより、特定のサブコントラクトへのdelegationを許可することができます。  
ここでは、subcidをDGALに指定することにより、subcidへのdelegationを許可します。

In [10]:
var resp = await rpc.call(adminWallet, cid_dlg, {}, { DGAL: [subcid] });
console.log(resp);

{
  txno: 701934,
  txid: 'xzvW66Fg6vDTvUSVDbYoCPtsxwXej7XZ5ReR4PQ5XE8Vy',
  status: 'ok',
  value: [ 'u58281888', 'u58281888' ]
}
